In [4]:
# --- 标量 (Scalar / Order 0 Tensor) ---
# 只有一个孤立的数值

import numpy as np
s = np.array(5)
print("Scalar s:", s)
print("Scalar dimension s:", s.ndim)

Scalar s: 5
Scalar dimension s: 0


In [11]:
# --- 向量 (Vector / Order 1 Tensor) ---
# 一列或者一行数字。PDF 中提到，按惯例未加说明的向量通常指列向量 [cite: 21]。
# 在 NumPy 中，最基础的一维数组既不是行也不是列，它就是一个单纯的“序列”
x = np.array([1, 2, 3])
print("Vector x:", x)
print("Vector x dimension:", x.ndim)
print("Vector x shape:", x.shape)

Vector x: [1 2 3]
Vector x dimension: 1
Vector x shape: (3,)


In [13]:
# 如果你想严格按照 PDF 说的“列向量（Matrix with n rows and 1 column）”来写 [cite: 21]：
x_col = np.array([[1], [2], [3]])
print("Column vector x:", x_col)
print("Column vector x dimension:", x_col.ndim)
print("Column vector x shape:", x_col.shape)

Column vector x: [[1]
 [2]
 [3]]
Column vector x dimension: 2
Column vector x shape: (3, 1)


In [17]:
# --- 矩阵 (Matrix / Order 2 Tensor) ---
# 有行有列的二维表格 [cite: 19]
A = np.array([[4, -5], [-2, 3]])
print("Matrix a:", A)
print("Matrix a dimension:", A.ndim)
print("Matrix a shape:", A.shape)

Matrix a: [[ 4 -5]
 [-2  3]]
Matrix a dimension: 2
Matrix a shape: (2, 2)


In [26]:
# Inner/Dot Product
x = np.array([1, 2, 3])
y = np.array([4, 5, 6])
# Dot Product, use np.dot()
dot_product_A = np.dot(x, y)
print("Dot Product A:", dot_product_A)
# Dot Product, use Python @
dot_product_B = x @ y
print("Dot Product B:", dot_product_B)
# Dot Product, use Python *, .sum()
dot_product_C = (x * y).sum()
print("Dot Product C:", dot_product_C)

Dot Product A: 32
Dot Product B: 32
Dot Product C: 32


In [28]:
# Outer Product
x = np.array([1, 2, 3])
y = np.array([4, 5])
outer_product_A = np.outer(x, y)
print("\n outer product: \n", outer_product_A)
# 输出：
# [[ 4  5]     <- 1*4, 1*5
#  [ 8 10]     <- 2*4, 2*5
#  [12 15]]    <- 3*4, 3*5
print("outer product shape:", outer_product_A.shape)


 outer product: 
 [[ 4  5]
 [ 8 10]
 [12 15]]
outer product shape: (3, 2)


In [34]:
from einops import einsum
x = np.array([1, 2, 3])
y = np.array([4, 5, 6])
# 1. 用 einsum 做点积 (对应数学中的求和 index i) 
# "i, i -> " 表示输入两个一维向量，它们的轴都叫 i。
# 箭头右边为空，意味着把 i 给消除（求和加起来），输出一个标量。
dot_ein = einsum(x, y, "i, i ->")
print("dot product:", dot_ein)
# 2. 用 einsum 做外积 (对应数学中的 i 和 j 分立) [cite: 48]
# "i, j -> i j" 表示输入向量 x(轴i) 和 y(轴j)
# 箭头右边保留了 i 和 j，意味着不求和，保留成一个二维矩阵。
outer_ein = einsum(x, np.array([4, 5]), "i, j -> i j")
print("outer product:", outer_ein)

dot product: 32
outer product: [[ 4  5]
 [ 8 10]
 [12 15]]


In [37]:
# Matrix Tranpose
# 创建一个 2 行 3 列的矩阵 A
A = np.array([[1, 2, 3],
              [4, 5, 6]])
# --- 方法 1：NumPy 自带的快捷方式 .T ---
A_trans_numpy = A.T
print("A tranpose:", A_trans_numpy)
# --- 方法 2：用你的 einsum 魔法 ---
# 把行（i）和列（j）的顺序反过来写
A_trans_ein = einsum(A, "i j -> j i")
print("A tranpose:", A_trans_ein)

A tranpose: [[1 4]
 [2 5]
 [3 6]]
A tranpose: [[1 4]
 [2 5]
 [3 6]]


In [44]:
# Matrix-Vector Product y = Ax
A = np.array([[1, 2, 3], [4, 5, 6]])
x = np.array([10, 20, 30])
# --- 传统视角：直接用 @ 符号（标准矩阵乘法） ---
y_standard = A @ x 
print("标准相乘结果 y = Ax:", y_standard)      # 输出 [140 320]
# --- einsum 视角：i表示A的行，j表示A的列，也是x的元素 ---
# 左右都有 j，说明 A 的列和 x 的元素相乘；右边没有 j，说明在列上求和消灭它，留下行（i）
y_ein = einsum(A, x, "i j, j -> i")
print("einsum 相乘结果:", y_ein)
# --- 斯坦福讲义视角：列的线性组合 ---
# A[:, 0] 代表第 0 列，x[0] 是它的权重 [cite: 28, 58]
y_column_view = x[0] * A[:, 0] + x[1] * A[:, 1] + x[2] * A[:, 2]
print("讲义‘列组合’验证:", y_column_view)     # 依然输出 [140 320]！
# 感觉建立： 当你看到 x[0] * A[:, 0] + ... 的结果和 A @ x 一模一样时，你脑海里的矩阵乘法就不再是死板的“第几行乘第几列”了，而是向量的按比例放大缩小再相加！这在机器学习中叫特征组合。

标准相乘结果 y = Ax: [140 320]
einsum 相乘结果: [140 320]
讲义‘列组合’验证: [140 320]


In [45]:
# y_ein = einsum(A, x, "i j, j -> i") 内部实现
A = np.array([[1, 2, 3], [4, 5, 6]])
x = np.array([10, 20, 30])
# ==========================================
# 计算机收到 "i j, j -> i" 后，在底层默默执行的代码：
# ==========================================

# 1. 根据箭头右边的 "-> i"，初始化一个全零的输出向量 y
# 因为 A 有 2 行 (i=0,1)，所以 y 的长度是 2
y_bottom = np.zeros(A.shape[0])
for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        # 3. 核心动作：对应位置相乘，并因为 j 在右边消失了，执行 += 累加
        y_bottom[i] += A[i][j] * x[j]
print("einsum 底层循环跑出来的结果:", y_bottom) # 输出 [140. 320.]

einsum 底层循环跑出来的结果: [140. 320.]


In [56]:
# Matrix-Matrix Product / C = AB
A = np.array([[1, 2, 3],
              [4, 5, 6]])

B = np.array([[7, 8],
              [9, 1],
              [2, 3]])
# --- 方法 1：标准 NumPy 乘法 ---
C_standard = A @ B
print("标准矩阵乘法 C = AB:\n", C_standard)


# --- 方法 2：einsum 填字游戏 ---
# 轴命名：A是 i行k列；B是 k行j列； [cite: 38]
# k 是公共轴（A的列数等于B的行数），相乘并在结果中把 k 消灭 [cite: 38, 39]
C_ein = einsum(A, B, "i k, k j -> i j")
print("\neinsum 矩阵乘法:\n", C_ein)

# --- 方法 3：讲义 2.3 提到的“trickier interpretation”（更高级的视角） ---
# C 实际上等于 A 的每一列，和 B 的每一行，做外积（Outer Product）之后，再全部加起来！[cite: 90, 91, 92]
# 数学公式：C = sum( a^1 * b_1^T + a^2 * b_2^T + ... ) [cite: 91]
C_outer_sum = np.outer(A[:, 0], B[0, :]) + np.outer(A[:, 1], B[1, :]) + np.outer(A[:, 2], B[2, :])
print("\n讲义‘外积之和’视角验证:\n", C_outer_sum)

标准矩阵乘法 C = AB:
 [[31 19]
 [85 55]]

einsum 矩阵乘法:
 [[31 19]
 [85 55]]

讲义‘外积之和’视角验证:
 [[31 19]
 [85 55]]


In [51]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])

B = np.array([[7, 8],
              [9, 1],
              [2, 3]])
C_bottom = np.zeros((A.shape[0], B.shape[1]))
for i in range(A.shape[0]):
    for k in range(A.shape[1]):
        for j in range(B.shape[1]):
            # 3. 核心动作：对应位置相乘，并因为 k 在右边消失了，执行 += 累加
            C_bottom[i][j] += A[i][k] * B[k][j]
print("einsum 底层循环跑出来的结果:", C_bottom)

einsum 底层循环跑出来的结果: [[31. 19.]
 [85. 55.]]
